In [ ]:
# Figure 3A–D: bilateral tracheole symmetry
# Inputs: 117 CATMAID skeletons, bilateral FlyWire neuropil meshes.
# Analysis: 200 nm centerline resampling, reflected/registered left neuropils, nearest-neighbor comparison.
from pathlib import Path
import math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import trimesh
from scipy.spatial import cKDTree
from scipy.stats import wilcoxon, spearmanr, rankdata
from matplotlib.collections import PolyCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from fafbseg import xform
import ncollpyde
from matplotlib.ticker import LogLocator

data_dir = Path('data')
node_file = data_dir / 'catmaid_nodes.csv'
mesh_dir = data_dir / 'neuropil_mesh'
fraction_file = Path('fig2_outputs') / 'tracheation_by_neuropil.csv'
out_dir = Path('fig3_outputs')
out_dir.mkdir(parents=True, exist_ok=True)
centerline_spacing_nm = 200.
point_chunk = 300_000
surface_samples = 15_000
icp_iterations = 40
icp_trim = 0.85
icp_tolerance_nm = 5.
min_spatial_points = 20
coverage_um = [2., 5.]
registration_seed = 20260903
excluded_bases = {'LA'}
skid_fields = ['skeleton_id', 'new_skeleton_id', 'skid', 'neuron']
node_fields = ['node_id', 'treenode_id', 'id']
parent_fields = ['parent_id', 'parent_node_id', 'parent']
x_fields = ['x_nm', 'x']; y_fields = ['y_nm', 'y']; z_fields = ['z_nm', 'z']

def normalise_name(value):
    return str(value).strip().lower().replace(' ', '_').replace('-', '_')

def resolve_column(columns, candidates):
    lookup = {normalise_name(column): column for column in columns}
    for candidate in candidates:
        key = normalise_name(candidate)
        if key in lookup:
            return lookup[key]

def load_trachea_nodes(path):
    raw = pd.read_csv(path, low_memory=False)
    skid_col = resolve_column(raw.columns, skid_fields)
    node_col = resolve_column(raw.columns, node_fields)
    parent_col = resolve_column(raw.columns, parent_fields)
    x_col = resolve_column(raw.columns, x_fields)
    y_col = resolve_column(raw.columns, y_fields)
    z_col = resolve_column(raw.columns, z_fields)
    nodes = pd.DataFrame({'skeleton_id': pd.to_numeric(raw[skid_col], errors='coerce'), 'node_id': pd.to_numeric(raw[node_col], errors='coerce'), 'parent_id': pd.to_numeric(raw[parent_col], errors='coerce'), 'x_nm': pd.to_numeric(raw[x_col], errors='coerce'), 'y_nm': pd.to_numeric(raw[y_col], errors='coerce'), 'z_nm': pd.to_numeric(raw[z_col], errors='coerce')})
    nodes = nodes.dropna(subset=['skeleton_id', 'node_id', 'x_nm', 'y_nm', 'z_nm']).copy()
    nodes['skeleton_id'] = nodes['skeleton_id'].astype(np.int64)
    nodes['node_id'] = nodes['node_id'].astype(np.int64)
    nodes['parent_id'] = pd.to_numeric(nodes['parent_id'], errors='coerce').astype('Int64')
    nodes = nodes.drop_duplicates(['skeleton_id', 'node_id']).reset_index(drop=True)
    return nodes

def transform_nodes_to_flywire(nodes):
    xyz_fafb = nodes[['x_nm', 'y_nm', 'z_nm']].to_numpy(dtype=np.float64)
    xyz_fw = xform.fafb14_to_flywire(xyz_fafb)
    if hasattr(xyz_fw, 'values'):
        xyz_fw = xyz_fw.values
    xyz_fw = np.asarray(xyz_fw, dtype=np.float64)
    out = nodes.copy()
    out[['x_fw_nm', 'y_fw_nm', 'z_fw_nm']] = xyz_fw
    return out

def resample_all_edges(nodes, spacing_nm):
    sampled_points = []
    sampled_weights = []
    sampled_skids = []
    total_exact_length_nm = 0.0
    edge_count = 0
    missing_parent_edges = 0
    for skid, group in nodes.groupby('skeleton_id', sort=False):
        node_ids = group['node_id'].to_numpy(dtype=np.int64)
        xyz = group[['x_fw_nm', 'y_fw_nm', 'z_fw_nm']].to_numpy(dtype=np.float64)
        lookup = {int(node_id): xyz[i] for i, node_id in enumerate(node_ids)}
        for row in group.itertuples(index=False):
            if pd.isna(row.parent_id):
                continue
            parent = lookup.get(int(row.parent_id))
            if parent is None:
                missing_parent_edges += 1
                continue
            child = np.array([row.x_fw_nm, row.y_fw_nm, row.z_fw_nm], dtype=np.float64)
            vector = child - parent
            length_nm = float(np.linalg.norm(vector))
            if not np.isfinite(length_nm) or length_nm <= 0:
                continue
            edge_count += 1
            total_exact_length_nm += length_nm
            n_parts = max(int(math.ceil(length_nm / spacing_nm)), 1)
            piece_length_nm = length_nm / n_parts
            mid_fractions = (np.arange(n_parts, dtype=np.float64) + 0.5) / n_parts
            points = parent[None, :] + mid_fractions[:, None] * vector[None, :]
            sampled_points.append(points)
            sampled_weights.append(np.full(n_parts, piece_length_nm, dtype=np.float64))
            sampled_skids.append(np.full(n_parts, int(skid), dtype=np.int64))
    points = np.vstack(sampled_points).astype(np.float64, copy=False)
    weights = np.concatenate(sampled_weights)
    skids = np.concatenate(sampled_skids)
    metadata = {'n_skeletons': int(nodes['skeleton_id'].nunique()), 'edge_count': int(edge_count), 'missing_parent_edges': int(missing_parent_edges), 'sample_point_count': int(len(points)), 'total_length_um': total_exact_length_nm / 1000.0}
    return (points, weights, skids, metadata)

def load_mesh_npz(name):
    path = mesh_dir / f'{name}.npz'
    with np.load(path, allow_pickle=False) as z:
        vertices = np.asarray(z['vertices'], dtype=np.float64)
        faces = np.asarray(z['faces'], dtype=np.int64)
    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    return mesh

def available_bilateral_pairs():
    names = {p.stem for p in mesh_dir.glob('*.npz')}
    left_bases = {name[:-2] for name in names if name.endswith('_L')}
    right_bases = {name[:-2] for name in names if name.endswith('_R')}
    bases = sorted((left_bases & right_bases) - excluded_bases)
    return bases

def points_inside_mesh(points_nm, mesh):
    points_nm = np.asarray(points_nm, dtype=np.float64)
    out = np.zeros(len(points_nm), dtype=bool)
    if not len(points_nm):
        return out
    bounds = np.asarray(mesh.bounds, dtype=np.float64)
    bbox_mask = np.all(points_nm >= bounds[0][None, :], axis=1) & np.all(points_nm <= bounds[1][None, :], axis=1)
    candidate_index = np.flatnonzero(bbox_mask)
    if not len(candidate_index):
        return out
    candidate_points = points_nm[candidate_index]
    collider = ncollpyde.Volume(np.asarray(mesh.vertices, dtype=np.float64), np.asarray(mesh.faces, dtype=np.int64), n_rays=3)
    inside_candidate = np.zeros(len(candidate_points), dtype=bool)
    for start in range(0, len(candidate_points), point_chunk):
        stop = min(start + point_chunk, len(candidate_points))
        inside_candidate[start:stop] = np.asarray(collider.contains(candidate_points[start:stop]), dtype=bool)
    out[candidate_index] = inside_candidate
    return out

def mesh_bbox_centre(mesh):
    bounds = np.asarray(mesh.bounds, dtype=np.float64)
    return (bounds[0] + bounds[1]) / 2.0

def determine_lateral_axis_and_midline(bases):
    pair_centres = []
    for base in bases:
        left = load_mesh_npz(f'{base}_L')
        right = load_mesh_npz(f'{base}_R')
        left_centre = mesh_bbox_centre(left)
        right_centre = mesh_bbox_centre(right)
        pair_centres.append((base, left_centre, right_centre))
    differences = np.vstack([np.abs(right - left) for _, left, right in pair_centres])
    median_difference = np.median(differences, axis=0)
    lateral_axis = int(np.argmax(median_difference))
    pair_midlines = np.array([(left[lateral_axis] + right[lateral_axis]) / 2.0 for _, left, right in pair_centres], dtype=np.float64)
    global_midline_nm = float(np.median(pair_midlines))
    return (lateral_axis, global_midline_nm, pair_centres)

def reflect_points(points, axis, plane_nm):
    reflected = np.asarray(points, dtype=np.float64).copy()
    reflected[:, axis] = 2.0 * plane_nm - reflected[:, axis]
    return reflected

def apply_transform(points, matrix):
    points = np.asarray(points, dtype=np.float64)
    homogeneous = np.column_stack([points, np.ones(len(points), dtype=np.float64)])
    return (matrix @ homogeneous.T).T[:, :3]

def rigid_kabsch(source, target):
    source = np.asarray(source, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    source_centre = source.mean(axis=0)
    target_centre = target.mean(axis=0)
    source_zero = source - source_centre
    target_zero = target - target_centre
    h = source_zero.T @ target_zero
    u, _, vt = np.linalg.svd(h)
    rotation = vt.T @ u.T
    if np.linalg.det(rotation) < 0:
        vt[-1, :] *= -1
        rotation = vt.T @ u.T
    translation = target_centre - rotation @ source_centre
    transform = np.eye(4, dtype=np.float64)
    transform[:3, :3] = rotation
    transform[:3, 3] = translation
    return transform

def sample_mesh_surface(mesh, n, seed):
    rng_state = np.random.get_state()
    np.random.seed(seed)
    try:
        points, _ = trimesh.sample.sample_surface(mesh, n)
    finally:
        np.random.set_state(rng_state)
    return np.asarray(points, dtype=np.float64)

def rigid_icp(source, target):
    source = np.asarray(source, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    transform = np.eye(4, dtype=np.float64)
    initial_translation = target.mean(axis=0) - source.mean(axis=0)
    transform[:3, 3] = initial_translation
    current = apply_transform(source, transform)
    target_tree = cKDTree(target)
    previous_median = np.inf
    for _ in range(icp_iterations):
        distances, indices = target_tree.query(current, k=1, workers=-1)
        cutoff = np.quantile(distances, icp_trim)
        keep = distances <= cutoff
        if keep.sum() < 20:
            break
        step = rigid_kabsch(current[keep], target[indices[keep]])
        current = apply_transform(current, step)
        transform = step @ transform
        new_distances, _ = target_tree.query(current, k=1, workers=-1)
        new_median = float(np.median(new_distances))
        if abs(previous_median - new_median) < icp_tolerance_nm:
            break
        previous_median = new_median
    final_distances, _ = target_tree.query(current, k=1, workers=-1)
    return (transform, current, final_distances)

def rotation_angle_deg(transform):
    rotation = np.asarray(transform[:3, :3], dtype=np.float64)
    cosine = (np.trace(rotation) - 1.0) / 2.0
    cosine = np.clip(cosine, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))

def symmetric_nn_metrics(left_points, right_points):
    left_points = np.asarray(left_points, dtype=np.float64)
    right_points = np.asarray(right_points, dtype=np.float64)
    if len(left_points) < min_spatial_points or len(right_points) < min_spatial_points:
        return None
    right_tree = cKDTree(right_points)
    left_tree = cKDTree(left_points)
    left_to_right, _ = right_tree.query(left_points, k=1, workers=-1)
    right_to_left, _ = left_tree.query(right_points, k=1, workers=-1)
    result = {'left_to_right_median_um': float(np.median(left_to_right)) / 1000.0, 'right_to_left_median_um': float(np.median(right_to_left)) / 1000.0, 'left_to_right_mean_um': float(np.mean(left_to_right)) / 1000.0, 'right_to_left_mean_um': float(np.mean(right_to_left)) / 1000.0, 'left_to_right_q95_um': float(np.quantile(left_to_right, 0.95)) / 1000.0, 'right_to_left_q95_um': float(np.quantile(right_to_left, 0.95)) / 1000.0}
    result['symmetric_median_nn_um'] = (result['left_to_right_median_um'] + result['right_to_left_median_um']) / 2.0
    result['symmetric_mean_nn_um'] = (result['left_to_right_mean_um'] + result['right_to_left_mean_um']) / 2.0
    result['symmetric_q95_nn_um'] = (result['left_to_right_q95_um'] + result['right_to_left_q95_um']) / 2.0
    for tolerance_um in coverage_um:
        tolerance_nm = tolerance_um * 1000.0
        left_coverage = float(np.mean(left_to_right <= tolerance_nm))
        right_coverage = float(np.mean(right_to_left <= tolerance_nm))
        result[f'coverage_{tolerance_um:g}um'] = (left_coverage + right_coverage) / 2.0
    return result

def unique_voxels(points, voxel_size_nm):
    if not len(points):
        return np.empty((0, 3), dtype=np.int64)
    voxels = np.floor(np.asarray(points, dtype=np.float64) / voxel_size_nm).astype(np.int64)
    return np.unique(voxels, axis=0)

def row_view(array):
    array = np.ascontiguousarray(array)
    return array.view(np.dtype((np.void, array.dtype.itemsize * array.shape[1]))).reshape(-1)

def occupancy_dice(points_a, points_b, voxel_size_um):
    voxel_size_nm = voxel_size_um * 1000.0
    a = unique_voxels(points_a, voxel_size_nm)
    b = unique_voxels(points_b, voxel_size_nm)
    if len(a) == 0 or len(b) == 0:
        return np.nan
    av = row_view(a)
    bv = row_view(b)
    intersection = len(np.intersect1d(av, bv, assume_unique=True))
    return float(2.0 * intersection / (len(a) + len(b)))

def analyse_pair(base, all_points, all_weights, all_skids, lateral_axis, midline_nm, return_details=False):
    left_name = f'{base}_L'
    right_name = f'{base}_R'
    left_mesh = load_mesh_npz(left_name)
    right_mesh = load_mesh_npz(right_name)
    left_mask = points_inside_mesh(all_points, left_mesh)
    right_mask = points_inside_mesh(all_points, right_mesh)
    left_points = all_points[left_mask]
    right_points = all_points[right_mask]
    left_weights = all_weights[left_mask]
    right_weights = all_weights[right_mask]
    left_skids = all_skids[left_mask]
    right_skids = all_skids[right_mask]
    left_length_um = float(left_weights.sum()) / 1000.0
    right_length_um = float(right_weights.sum()) / 1000.0
    total_lr_length = left_length_um + right_length_um
    length_ai = (right_length_um - left_length_um) / total_lr_length if total_lr_length > 0 else np.nan
    reflected_left = reflect_points(left_points, lateral_axis, midline_nm)
    raw_metrics = symmetric_nn_metrics(reflected_left, right_points)
    seed = registration_seed + sum((ord(c) for c in base))
    left_surface = sample_mesh_surface(left_mesh, surface_samples, seed)
    right_surface = sample_mesh_surface(right_mesh, surface_samples, seed + 1)
    reflected_left_surface = reflect_points(left_surface, lateral_axis, midline_nm)
    registration, registered_left_surface, surface_distances = rigid_icp(reflected_left_surface, right_surface)
    registered_left_points = apply_transform(reflected_left, registration)
    registered_metrics = symmetric_nn_metrics(registered_left_points, right_points)
    left_diag_um = float(np.linalg.norm(np.ptp(np.asarray(left_mesh.bounds, dtype=np.float64), axis=0)) / 1000.0)
    right_diag_um = float(np.linalg.norm(np.ptp(np.asarray(right_mesh.bounds, dtype=np.float64), axis=0)) / 1000.0)
    region_scale_um = (left_diag_um + right_diag_um) / 2.0
    row = {'region': base, 'left_neuropil': left_name, 'right_neuropil': right_name, 'left_tlc_length_um': left_length_um, 'right_tlc_length_um': right_length_um, 'total_lr_tlc_length_um': total_lr_length, 'length_asymmetry_index_R_minus_L_over_sum': length_ai, 'absolute_length_asymmetry': abs(length_ai) if np.isfinite(length_ai) else np.nan, 'left_sample_points': int(len(left_points)), 'right_sample_points': int(len(right_points)), 'left_contributing_tracheoles': int(len(np.unique(left_skids))), 'right_contributing_tracheoles': int(len(np.unique(right_skids))), 'region_scale_um': region_scale_um, 'neuropil_registration_median_surface_error_um': float(np.median(surface_distances)) / 1000.0, 'neuropil_registration_q95_surface_error_um': float(np.quantile(surface_distances, 0.95)) / 1000.0, 'registration_rotation_deg': rotation_angle_deg(registration)}
    if raw_metrics is None:
        row.update({'raw_mirror_symmetric_median_nn_um': np.nan, 'raw_mirror_symmetric_mean_nn_um': np.nan, 'raw_mirror_symmetric_q95_nn_um': np.nan})
    else:
        row.update({'raw_mirror_symmetric_median_nn_um': raw_metrics['symmetric_median_nn_um'], 'raw_mirror_symmetric_mean_nn_um': raw_metrics['symmetric_mean_nn_um'], 'raw_mirror_symmetric_q95_nn_um': raw_metrics['symmetric_q95_nn_um']})
    if registered_metrics is None:
        row.update({'registered_symmetric_median_nn_um': np.nan, 'registered_symmetric_mean_nn_um': np.nan, 'registered_symmetric_q95_nn_um': np.nan, 'registered_normalized_median_nn': np.nan})
        for tolerance_um in coverage_um:
            row[f'registered_coverage_{tolerance_um:g}um'] = np.nan
    else:
        median_nn = registered_metrics['symmetric_median_nn_um']
        row.update({'registered_symmetric_median_nn_um': median_nn, 'registered_symmetric_mean_nn_um': registered_metrics['symmetric_mean_nn_um'], 'registered_symmetric_q95_nn_um': registered_metrics['symmetric_q95_nn_um'], 'registered_normalized_median_nn': median_nn / region_scale_um if region_scale_um > 0 else np.nan})
        for tolerance_um in coverage_um:
            row[f'registered_coverage_{tolerance_um:g}um'] = registered_metrics[f'coverage_{tolerance_um:g}um']
    details = None
    if return_details:
        details = {'left_mesh': left_mesh, 'right_mesh': right_mesh, 'left_points': left_points, 'right_points': right_points, 'reflected_left_points': reflected_left, 'registered_left_points': registered_left_points, 'registration': registration}
    return (row, details)

# Recompute the bilateral measurements from the same source skeletons and meshes.
nodes = load_trachea_nodes(node_file)
nodes = transform_nodes_to_flywire(nodes)
points, weights, skids, meta = resample_all_edges(nodes, centerline_spacing_nm)
assert meta['n_skeletons'] == 117
bases = available_bilateral_pairs()
assert len(bases) == 34, f'Expected 34 bilateral neuropil pairs, found {len(bases)}.'
lateral_axis, midline_nm, _ = determine_lateral_axis_and_midline(bases)
rows, details_by_region = [], {}
for base in bases:
    row, detail = analyse_pair(base, points, weights, skids, lateral_axis, midline_nm, return_details=True)
    rows.append(row)
    details_by_region[base] = detail
symmetry = pd.DataFrame(rows)
symmetry[['region', 'left_tlc_length_um', 'right_tlc_length_um',
          'length_asymmetry_index_R_minus_L_over_sum', 'raw_mirror_symmetric_median_nn_um',
          'registered_symmetric_median_nn_um', 'registered_normalized_median_nn',
          'registered_coverage_2um', 'registered_coverage_5um']].to_csv(out_dir / 'bilateral_metrics.csv', index=False)

# Median bootstrap CIs and paired rank-biserial statistics.
n_boot = 5000
stat_seed = 20260909
rng = np.random.default_rng(stat_seed)
def bootstrap_median_ci(values, n_boot=n_boot):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    n = len(values)
    if n < 2:
        return (np.nan, np.nan)
    boots = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        sample = values[rng.integers(0, n, size=n)]
        boots[i] = np.median(sample)
    return tuple(np.quantile(boots, [0.025, 0.975]))

def paired_rank_biserial(differences):
    d = np.asarray(differences, dtype=np.float64)
    d = d[np.isfinite(d)]
    d = d[d != 0]
    if not len(d):
        return np.nan
    ranks = rankdata(np.abs(d), method='average')
    w_pos = np.sum(ranks[d > 0])
    w_neg = np.sum(ranks[d < 0])
    denom = w_pos + w_neg
    if denom == 0:
        return np.nan
    return float((w_pos - w_neg) / denom)

spatial = symmetry.dropna(subset=['raw_mirror_symmetric_median_nn_um', 'registered_symmetric_median_nn_um']).copy()
raw = spatial['raw_mirror_symmetric_median_nn_um'].to_numpy(float)
registered = spatial['registered_symmetric_median_nn_um'].to_numpy(float)
improvement = raw - registered
stat_rows = []
def record(metric, values, wilcoxon_test=False):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    ci = bootstrap_median_ci(values)
    row = {'metric': metric, 'n': len(values), 'median': float(np.median(values)), 'ci_low': ci[0], 'ci_high': ci[1]}
    if wilcoxon_test:
        test = wilcoxon(values, alternative='two-sided', zero_method='wilcox', method='auto')
        row.update(p=float(test.pvalue), paired_rank_biserial=paired_rank_biserial(values))
    stat_rows.append(row)
record('raw mismatch (µm)', raw)
record('registered mismatch (µm)', registered)
record('registration reduction (µm)', improvement, True)
record('normalized mismatch', spatial['registered_normalized_median_nn'])
record('5 µm local correspondence', spatial['registered_coverage_5um'])

# Figure 3D uses 34 pairs; exclude optic laminae from the 76-region volume table.
fractions = pd.read_csv(fraction_file)
fractions = fractions.loc[~fractions['neuropil'].isin(['LA_L', 'LA_R'])].copy()
assert len(fractions) == 76
fraction_map = fractions.set_index('neuropil')['tracheation_percent'].to_dict()
pair_rows = []
for base in bases:
    l, r = fraction_map[f'{base}_L'], fraction_map[f'{base}_R']
    pair_rows.append({'region': base, 'left': l, 'right': r, 'difference': r - l, 'asymmetry': (r - l) / (r + l)})
lr = pd.DataFrame(pair_rows)
record('right minus left fraction (percentage points)', lr['difference'], True)
record('signed volume-fraction asymmetry', lr['asymmetry'], True)
rho, p_rho = spearmanr(lr['left'], lr['right'])
stat_rows.append({'metric': 'left vs right fraction Spearman rho', 'n': len(lr), 'median': rho,
                  'ci_low': np.nan, 'ci_high': np.nan, 'p': p_rho})
pd.DataFrame(stat_rows).to_csv(out_dir / 'fig3_stats.csv', index=False)
print(pd.DataFrame(stat_rows).round(5).to_string(index=False))

# Figure 3A: global reflection versus region-specific registration.
bins = np.array([5., 10., 15.])
colors = ['#3B0F70', '#287D8E', '#55C667', '#DCE319']
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
for ax, mode in zip(axes, ['reflected_left_points', 'registered_left_points']):
    actual_all, near_all, distances_all = [], [], []
    for base in bases:
        d = details_by_region[base]
        actual, opposite = d['right_points'], d[mode]
        if not len(actual) or not len(opposite): continue
        chosen = np.linspace(0, len(actual)-1, min(12000, len(actual)), dtype=int)
        chosen_other = np.linspace(0, len(opposite)-1, min(12000, len(opposite)), dtype=int)
        sampled = actual[chosen]
        actual_all.append(sampled)
        near_all.append(opposite[chosen_other])
        distances_all.append(cKDTree(opposite).query(sampled, k=1)[0] / 1000)
    actual, counterpart = np.vstack(actual_all), np.vstack(near_all)
    dist = np.concatenate(distances_all)
    ax.scatter(counterpart[:, 0]/1000, counterpart[:, 1]/1000, s=.22, c='0.75', alpha=.20, rasterized=True)
    classes = np.digitize(dist, bins)
    for group in range(3, -1, -1):
        take = classes == group
        ax.scatter(actual[take,0]/1000, actual[take,1]/1000, s=.3, color=colors[group], alpha=.9, rasterized=True)
    ax.set_aspect('equal'); ax.set_axis_off()
    ax.set_title('Global reflection' if mode == 'reflected_left_points' else 'Region-registered reflection', fontsize=10)
fig.legend(handles=[Patch(color=c, label=lab) for c, lab in zip(colors, ['0–5 µm','5–10 µm','10–15 µm','>15 µm'])],
           ncol=4, loc='lower center', frameon=False)
fig.tight_layout(rect=(0,.06,1,1)); fig.savefig(out_dir / 'fig3a.svg', bbox_inches='tight'); plt.show()

# Figure 3B: representative high, typical and low correspondence.
regions = ['PVLP', 'SIP', 'LOP']
fig, axes = plt.subplots(2, len(regions), figsize=(12, 5.3))
for j, region in enumerate(regions):
    d = details_by_region[region]
    right, left, mesh = d['right_points'], d['registered_left_points'], d['right_mesh']
    faces = np.asarray(mesh.faces, int)
    if len(faces) > 3500: faces = faces[np.linspace(0, len(faces)-1, 3500, dtype=int)]
    tri = np.asarray(mesh.vertices)[faces][:, :, [0,2]] / 1000.
    residual = cKDTree(left).query(right, k=1)[0] / 1000.
    for i, ax in enumerate(axes[:, j]):
        ax.add_collection(PolyCollection(tri, facecolors='0.83', alpha=.14, edgecolors='none', rasterized=True))
        if i == 0:
            ax.scatter(right[:,0]/1000, right[:,2]/1000, s=.48, c='#D77F3C', alpha=.85, rasterized=True)
            ax.scatter(left[:,0]/1000, left[:,2]/1000, s=.48, c='#319A81', alpha=.85, rasterized=True)
        else:
            ax.scatter(right[:,0]/1000, right[:,2]/1000, s=.22, c='0.6', alpha=.25, rasterized=True)
            far = residual > 5.
            ax.scatter(right[far,0]/1000, right[far,2]/1000, s=.7, c='#B52B34', rasterized=True)
        extent = np.vstack([right[:, [0,2]], left[:, [0,2]], np.asarray(mesh.vertices)[:, [0,2]]]) / 1000.
        pad = .07 * np.ptp(extent, axis=0)
        ax.set_xlim(extent[:,0].min()-pad[0], extent[:,0].max()+pad[0])
        ax.set_ylim(extent[:,1].max()+pad[1], extent[:,1].min()-pad[1])
        ax.set_aspect('equal'); ax.set_axis_off()
    axes[0,j].set_title(f"{region} ({symmetry.set_index('region').loc[region, 'registered_coverage_5um']:.0%})")
axes[0,0].set_ylabel('Bilateral overlay'); axes[1,0].set_ylabel('>5 µm mismatch')
fig.tight_layout(); fig.savefig(out_dir / 'fig3b.svg', bbox_inches='tight'); plt.show()

# Figure 3C: length asymmetry, normalized spatial mismatch and local correspondence.
plot_df = symmetry.sort_values('region').reset_index(drop=True)
fig, axes = plt.subplots(1, 3, figsize=(11, 10), sharey=True)
y = np.arange(len(plot_df))
axes[0].scatter(plot_df['length_asymmetry_index_R_minus_L_over_sum'], y, color='0.3', s=18)
axes[0].axvline(0, ls=':', lw=.8, color='0.6'); axes[0].set_xlabel('(R − L) / (R + L)')
axes[1].scatter(plot_df['registered_normalized_median_nn'], y, color='0.3', s=18)
axes[1].set_xlabel('Normalized spatial mismatch')
axes[2].plot(plot_df['registered_coverage_2um'], y, 'o', ms=3, mfc='white', mec='0.4', label='2 µm')
axes[2].plot(plot_df['registered_coverage_5um'], y, 'o', ms=3, color='#369C92', label='5 µm')
axes[2].set_xlabel('Correspondence fraction'); axes[2].legend(frameon=False)
axes[0].set_yticks(y, plot_df['region']); axes[0].invert_yaxis()
for ax in axes: ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); fig.savefig(out_dir / 'fig3c.svg', bbox_inches='tight'); plt.show()

# Figure 3D: right versus left volume fraction for the 34 paired neuropils.
fig, ax = plt.subplots(figsize=(5, 4.8))
positive = lr[(lr['left'] > 0) & (lr['right'] > 0)].copy()
assert len(positive) == 34
highlight = {'AME', 'AOTU', 'MB_PED', 'GA', 'EPA'}
for _, row in positive.iterrows():
    special = row['region'] in highlight
    ax.scatter(row['left'], row['right'], s=30, color='#C46C4F' if special else '#526B8A', alpha=.88)
    if special: ax.annotate(row['region'], (row['left'], row['right']), xytext=(4,3), textcoords='offset points', fontsize=8)
limits = [min(positive[['left','right']].min())*.7, max(positive[['left','right']].max())*1.4]
ax.plot(limits, limits, ls='--', lw=.8, color='0.65'); ax.set(xscale='log', yscale='log', xlim=limits, ylim=limits)
ax.set_xlabel('Left tracheole volume fraction (%)'); ax.set_ylabel('Right tracheole volume fraction (%)')
ax.text(.04, .97, f'ρ = {rho:.2f}\nn = {len(lr)} pairs', va='top', transform=ax.transAxes, fontsize=9)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); fig.savefig(out_dir / 'fig3d.svg', bbox_inches='tight'); plt.show()